# Altair Practice Lab

In this assignment we will be crafting a series of visualizations using
Altair to get practice working with real data in this context.

Your responses should be within the functions given,
using appropriate helper functions to help with clarity
and reduce redundancy.

## Rubric

The criteria to receive an S on this assignment is a good-faith attempt at each portion.

A good-faith attempt should either:

- be fundamentally correct producing the expected output with minimal deviations,
- **OR** contain an explanation of what does not work and _details on what was tried_.

**To receive an N, at least half of the assignment should have good-faith attempts.**

Your charts do not need to match the examples exactly!  They are helpful to get a sense of what you're after, but focus on the problem description.

## Introducing the Dataset

In the data directory you'll find three files:

**legislators.csv** which consists of ~7400 records representing state legislators, it has the following fields:

- name
- given_name
- family_name
- party: As reported by the state.
- gender: Male / Female / Other\*
- jurisdiction: This field contains an identifier for the state or jurisdiction, see below for details.
- district: the name of the district represented
- type: upper | lower - The classification of the legislative. Most states have both, but DC and NE only have an upper chamber.

Note: Accurate data on gender is hard to come by in many states. There may be irregularities in this field. This is also why this field does not make further distinctions beyond Male/Female/Other.

**populations.csv** 

A CSV file with a row with each state's population.  

**actions.csv**

A CSV file with legislative actions for a few states across a range of years:

- description: action text as reported by state
- date: date of action recorded
- classification: not used for this assignment
- state: 2-letter state postal code (note: these are upper case, other files use lower case)
- session: an identifier representing which session the action comes from (not used)

In [21]:
import altair as alt
import polars as pl
import pathlib
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

## Part 1: Data Exploration

First, we'll build a few exploratory visualizations to get a sense of the data for this assignment.

### 1.0: Cleaning

As mentioned above, there is no 'state' field. Jurisdiction is in the format:

`ocd-jurisdiction/country:us/state:nc/government`

and for non-states:

- `ocd-jurisdiction/country:us/district:dc/government`
- `ocd-jurisdiction/country:us/territory:pr/government`

So for our purposes, we want to add a `state` column from the two letter code after either "state:", "district:", or "territory:".
(We will treat DC and PR as states.)

Complete the function `legislators_df` which should return the data from `legislators.csv` in a dataframe, with an additional `state` column.

In [22]:
def legislators_df():
    '''
    Imports the legislators file and add state information to it.

    Input: 
        - path (str): Path to the legislators file

    Output: Polars dataframe
    '''
    # Load file from path
    # Ref for fighting with pathlib: https://docs.python.org/3/library/pathlib.html#basic-use
    root = pathlib.Path().resolve()
    filepath = root / "data" / "legislators.csv"

    # Import polars dataframe
    df = pl.read_csv(filepath)

    # Extract state
    # Ref for polars regex implementation: https://docs.rs/regex/latest/regex/#syntax
    # Used polars docs for syntax of polars: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.with_columns.html
    df = df.with_columns(
        pl.col('jurisdiction').str.extract(r"(state|district|territory):([a-z]{2})/", 2).
        alias('state')
    )
    df = df.drop('jurisdiction')

    # Import path
    return df

# render dataframe
legislators_df()

name,given_name,family_name,gender,party,district,type,state
str,str,str,str,str,str,str,str
"""Robert White""","""Robert C.""","""White""","""Male""","""Independent""","""At-Large""","""legislature""","""dc"""
"""Charles Allen""","""Charles""","""Allen""","""Male""","""Democratic""","""Ward 6""","""legislature""","""dc"""
"""Vince Gray""","""Vincent Condol""","""Gray""","""Male""","""Democratic""","""Ward 7""","""legislature""","""dc"""
"""Phil Mendelson""","""Philip Heath""","""Mendelson""","""Male""","""Democratic""","""Chairman""","""legislature""","""dc"""
"""Trayon White""","""Trayon""","""White""","""Male""","""Democratic""","""Ward 8""","""legislature""","""dc"""
…,…,…,…,…,…,…,…
"""Sol Higgins""","""Sol Yamiz""","""Higgins Cuadrado""","""Female""","""Partido Popular Democrático""","""35""","""lower""","""pr"""
"""Migdalia González""","""Migdalia I.""","""Gonzalez Arroyo""","""Female""","""Partido Popular Democrático""","""4""","""upper""","""pr"""
"""Keren Riquelme""","""Keren L.""","""Riquelme Cabrera""","""Female""","""Partido Nuevo Progresista""","""At-Large""","""upper""","""pr"""


### 1.1: Initial Plot

First let's build a visualization of gender breakdowns in state legislatures.
Use the following:

- stacked bars per state
- each segment of stacked bar is gender

Your graph should somewhat resemble *imgs/ex1.1.png*.

In [23]:
def states_by_gender_initial(df):
    chart = alt.Chart(df).mark_bar().encode(
        x = 'state',
        y = 'count(gender)',
        color='gender'
    )  

    # Print chart
    chart 
    return chart   

# render chart
states_by_gender_initial(legislators_df())

alt.Chart(...)

### 1.2: Improvements

While it is clear from the first chart that there are more elected officials that are men than women, it is hard to compare across states.

Make the following adjustments:

- Normalize the chart so that each bar is a percentage, allowing for direct comparison across states.
- Since this is US political data, the colors red and blue have a strong meaning, associated with the Republican and Democratic parties. Change the color scheme to avoid red and blue. (I chose #8624f5 for women and #1fc3aa for men based on this article: <https://blog.datawrapper.de/gendercolor/>)
- Two states are very close to 50%, add a line at 50% using a layered chart to make it easier to see if they exceed 50% or not.

Your graph should somewhat resemble *imgs/ex1.2.png*.

In [24]:
def states_by_gender_improved(df):
    # Save colors
    # Ref: https://altair-viz.github.io/user_guide/customization.html
    domain_ = ['Male', "Female", "Other"]
    range_ = ["#1fc3aa", "#8624f5", "gray"]

    # Updated chart code
    # Ref: https://altair-viz.github.io/gallery/normalized_stacked_bar_chart.html
    chart = alt.Chart(df).mark_bar().encode(
        x = 'state',
        y = alt.Y('count(gender)').stack("normalize"),
        color=alt.Color('gender').scale(domain=domain_, range=range_)
    )  

    # Adding a horizontal line
    rule = alt.Chart(df).mark_rule(strokeDash=[2, 2]).encode(
        y=alt.datum(0.5),
        color=alt.value("black")
    )

    # Print chart
    chart + rule
    return chart + rule  

# render chart
states_by_gender_improved(legislators_df())

alt.LayerChart(...)

## Part 2: Party Breakdown

We'll now take a look at party control. We can start with essentially the same chart.

### 2.0 - Party Control

Copy your code from 1.2 above & modify it to use party instead of gender.  Your graph will wind up with too many parties, see `imgs/ex2.0.png`.

In [25]:
def party_control_raw(df):
    # Chart base
    chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x="state",
            y=alt.Y("count(party)").stack("normalize"),
            color=alt.Color("party"),
        )
    )

    # Adding a horizontal line
    rule = (
        alt.Chart(df)
        .mark_rule(strokeDash=[2, 2])
        .encode(y=alt.datum(0.5), color=alt.value("black"))
    )

    # Print chart
    chart + rule
    return chart + rule

# render chart
party_control_raw(legislators_df())

alt.LayerChart(...)

### 2.1 - Cleaning Data

The above graph still has some shortcomings:

- Most states have an upper and lower chamber, and party control may vary between them. We'll need to make two bars per state (which we'll tackle in 2.2).
- Also, there are too many variations of party as you can see here:

Let's transform the data again, adding a new column "party_code" with the following rules:

- if the word 'Democratic' appears, set party_code to 'D'
- if the word 'Republican' appears, set the party_code to 'R'
- otherwise, set the party_code to 'O'

Party data in NE, DC, and PR does not work with this scheme.
For simplicity, we will exclude them from our analysis.

For this portion, implement `clean_party_df` which should return a modified legislators DataFrame with the `party_code` column, and the rows for states 'DC', 'NE' and 'PR' dropped.

In [26]:
def clean_party_df():
    # start with the DataFrame from part 1 & return transformed copy
    df = legislators_df()

    # Clean strings
    # ref to figure out command: https://stackoverflow.com/a/75935390
    # ref: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.when.html
    # ref: to figure out how to coerce else into a string: https://github.com/pola-rs/polars/issues/13805#issuecomment-1898428251
    df = df.with_columns(
        pl.when(pl.col("party").str.contains(r"(?i)Democratic"))
        .then(pl.lit("D"))
        .when(pl.col("party").str.contains(r"(?i)Republican"))
        .then(pl.lit("R"))
        .otherwise(pl.lit("O"))
        .alias("party_code")
    )

    # Drop rows with states in list
    drop_lst = ['dc', 'ne', 'pr']
    df = df.filter(~pl.col('state').is_in(drop_lst))

    return df

clean_party_df()

name,given_name,family_name,gender,party,district,type,state,party_code
str,str,str,str,str,str,str,str,str
"""Kevin McCabe""","""Kevin J.""","""McCabe""","""Male""","""Republican""","""30""","""lower""","""ak""","""R"""
"""C.J. McCormick""","""Conrad J.""","""McCormick""","""Male""","""Democratic""","""38""","""lower""","""ak""","""D"""
"""Bill Wielechowski""","""Bill P.""","""Wielechowski""","""Male""","""Democratic""","""K""","""upper""","""ak""","""D"""
"""Justin Ruffridge""","""Justin""","""Ruffridge""","""Male""","""Republican""","""7""","""lower""","""ak""","""R"""
"""Will Stapp""","""Will""","""Stapp""","""Male""","""Republican""","""32""","""lower""","""ak""","""R"""
…,…,…,…,…,…,…,…,…
"""Karlee Provenza""","""Karlee R.""","""Provenza""","""Female""","""Democratic""","""45""","""lower""","""wy""","""D"""
"""Affie Ellis""","""Affie Burnside""","""Ellis""","""Female""","""Republican""","""8""","""upper""","""wy""","""R"""
"""Abby Angelos""","""Abby""","""Angelos""","""Female""","""Republican""","""3""","""lower""","""wy""","""R"""


### 2.2 - Faceted Plot

Add a function `party_control_by_chamber` that contains the following elements:

- One bar per state, **along the Y axis**.
- Each bar should consist of a stack: a blue portion, a green portion, and a red portion, corresponding to the D, O, and R `party_code` respectively.
- A vertical line at the 50% mark, indicating (likely) party control.
- Finally, facet the chart on `type` so that you get a set of bars for the lower and upper chambers.

See `imgs/ex2.2.png` for an example.

In [27]:
def party_control_by_chamber(df):
    # Save colors
    domain_ = ["D", "O", "R"]
    range_ = ["blue", "green", "red"]

    # Save chart
    chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x="state",
            y=alt.Y("count(party_code)").stack("normalize"),
            color=alt.Color("party_code").scale(domain=domain_, range=range_)
        )
    )

    # Add 50% line
    rule = (
        alt.Chart(df)
        .mark_rule(strokeDash=[2, 2])
        .encode(y=alt.datum(0.5), color=alt.value("black"))
    )

    # Print output
    grph = (chart + rule).facet(row="type")
    return grph

# Call plot
party_control_by_chamber(clean_party_df())

alt.FacetChart(...)

## Part 3: Comparing by Population

For part three, we are interested in the relationship of various properties of legislatures to the total population of the state.

To do this, we'll need to create a combined DataFrame that mixes in data from `populations.csv`.

### 3.0 - Create Combined DataFrame

Write the function `population_combined_df`, which should return a DataFrame with the columns:

- state: abbreviation of state
- upper: total seats in upper chamber
- lower: total seats in lower chamber
- pop_2020: the 2020 population, obtained from merging with `population.csv`

**Data Note:** These numbers are based on the non-vacant seats as-of a particular day in September 2024. Vacancies will cause the counts to be off by a bit, but the general size of the legislature should be roughly the same.

In [28]:
def population_combined_df():
    '''
    This function loads and merges the population data to the legislators 
    dataset.
    '''
    # Load population dataset
    root = pathlib.Path().resolve()
    filepath = root / "data" / "populations.csv"

    # Import both datasets at their own units of analysis
    pop = pl.read_csv(filepath)
    leg = legislators_df()

    # Cut legislators to relevant cols, then collapse
    # Ref: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.len.html#polars.dataframe.group_by.GroupBy.len
    leg = leg.select('state', 'type')
    leg = leg.group_by('state', 'type').len(name='count')
    leg = leg.pivot("type", index="state", values="count")
    leg = leg.drop('legislature') 

    # Merge
    # Ref: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join.html
    df = leg.join(pop, on='state', validate='1:1')

    return df

population_combined_df()

state,lower,upper,pop_2020
str,u32,u32,i64
"""al""",104,35,5024294
"""ak""",40,20,733374
"""az""",59,30,7157902
"""ar""",98,35,3011490
"""ca""",79,40,39538212
…,…,…,…
"""wa""",98,49,7705267
"""wv""",99,34,1793713
"""wi""",98,32,5893713


### 3.1 - Create Population vs. Seats Scatterplot

Create a new plot with two layers:

- Population on the X axis
- Number of seats on the Y axis
- Upper chamber points should be purple and use the 'triangle-up' shape.
- Lower chamber points should be orange and use the 'triangle-down' shape.
- Make a customization or two to your chart's default labels and axes, whatever you feel is appropriate.

Hint: You can layer two charts for this.

In [29]:
def scatter_pop_size(df):
    # Create marks by group
    # Ref for title changes: https://altair-viz.github.io/user_guide/customization.html
    lower = (
        alt.Chart(
            df,
            title=alt.Title(
                "Lower and upper house seats tend to increase with population",
                subtitle="Upper (purple) and lower (orange) seats in each state",
                anchor="start",
                orient="top",
                offset=10
            ),
        )
        .mark_point(color="orange", shape="triangle-down")
        .encode(
            alt.X("pop_2020").title("Population (2020)"),
            alt.Y("lower").title("Number of seats"),
        )
    )
    upper = (
        alt.Chart(df)
        .mark_point(color="purple", shape="triangle-up")
        .encode(
            x="pop_2020",
            y="upper",
        )
    )

    # Stack graphs
    graph = (lower + upper).configure_axis(
        grid=False
    )

    # Add labeling
    return graph

# Render plot
scatter_pop_size(population_combined_df())

alt.LayerChart(...)

### 3.2 - Regressions

Add two more layers, a purple & orange regression line for each chamber.  See `imgs/ex3.2.png`

Hint: See `transform_regression`.

In [30]:
def scatter_pop_size_regression(df):
    '''
    Copying and pasting code here, which is not great. Sorry to duplicate
    instead of more elegantly handling adding the axis configurations.
    '''
    # Create marks by group
    # Ref for title changes: https://altair-viz.github.io/user_guide/customization.html
    lower = (
        alt.Chart(
            df,
            title=alt.Title(
                "Lower and upper house seats tend to increase with population",
                subtitle="Upper (purple) and lower (orange) seats in each state",
                anchor="start",
                orient="top",
                offset=10,
            ),
        )
        .mark_point(color="orange", shape="triangle-down")
        .encode(
            alt.X("pop_2020").title("Population (2020)"),
            alt.Y("lower").title("Number of seats"),
        )
    )
    upper = (
        alt.Chart(df)
        .mark_point(color="purple", shape="triangle-up")
        .encode(
            x="pop_2020",
            y="upper",
        )
    )

    # Add regressions
    reg_lower = lower.transform_regression("pop_2020", "lower").mark_line(color="orange")
    reg_upper = upper.transform_regression("pop_2020", "upper").mark_line(color="purple")

    # Stack graphs
    graph = (lower + upper + reg_lower + reg_upper).configure_axis(grid=False)

    # Add labeling
    return graph

# Render plot
scatter_pop_size_regression(population_combined_df())

alt.LayerChart(...)

## Part 4: Actions Heatmap

The file `actions_il-in-mi-wi_2021-2024.csv` contains nearly half a million records, representing every official action taken on a piece of legislation in these four midwestern states over the past two sessions.

Legislatures work quite differently, some meet all year, while others meet for very short periods.
By creating a heatmap of what days actions take place, we can get a sense of how different states compare.

### 4.0 - Load Actions

Complete `actions_df`, which should load the data from `actions_il-in-mi-wi_2021-2024.csv`.

Tips: 
- Make sure that the `date` column is loaded as a date type!
- Dates are in YYYY-MM-DD format, though some also have additional characters for time, which you will want to ignore.

In [31]:
def actions_df():
    '''
    Loads in the actions_df from a csv file and does some minor cleaning. Then,
    it converts the actions DF to the unit of analysis needed for the graphs
    (state days)
    '''
    # Load population dataset
    root = pathlib.Path().resolve()
    filepath = root / "data" / "actions_il-in-mi-wi_2021-2024.csv"

    # Import both datasets at their own units of analysis
    df = pl.read_csv(filepath)

    # Reformat date, extracting appropriate format first
    # Ref: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.to_date.html
    df = df.with_columns(
        pl.col('date').str.extract(r"([0-9]{4}-[0-9]{2}-[0-9]{2})")
        .alias('date_ymd')

    )

    # Collapse data, then convert date to a date
    df = df.drop("description", 'date', "classification", "session")
    df = df.group_by("state", "date_ymd").len(name="actions")
    df = df.with_columns(
        pl.col('date_ymd').str.to_date(strict = True)
    )

    # Return df
    return df

# Call functions
actions_df()

state,date_ymd,actions
str,date,u32
"""MI""",2022-09-14,20
"""WI""",2024-03-06,197
"""IL""",2023-03-29,783
"""IL""",2023-08-11,131
"""IN""",2023-05-04,182
…,…,…
"""IL""",2021-06-01,369
"""IN""",2023-02-02,213
"""IL""",2021-02-17,3829


### 4.1 - Actions Heatmap

Generate a heatmap (using `mark_rect`) with:

- a row per state
- each row consists of shaded marks with shading based on the total action count for a given week

Tip: Use the 'yearweek(date)' aggregation for the X channel.

See `imgs/ex4.1.png`.

In [32]:
def actions_heatmap_scaled(df):
    '''
    Trying a docstring to fix jupyter not rendering my charts
    '''
    # Create chart object
    chart = alt.Chart(df).mark_rect().encode(
        alt.X("yearweek(date_ymd):O"),
        alt.Y("state:N"),
        alt.Color("actions")
    )

    chart
    return chart

actions_heatmap_scaled(actions_df())

alt.Chart(...)

### 4.2 - Excluding IL Outliers

Illinois clearly dominates the above graph, below modify two calls to `actions_heatmap` with a modified dataframe that excludes IL, and a modified dataframe that only includes IL.

(Note how by using functions in our dataframe we can more easily reuse portions by making small adjustments to the data.)

See `ex4.2a.png` and `ex4.2b.png`

In [33]:
# Ref: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.filter.html
actions_no_il = actions_df().filter(pl.col('state') != 'IL')
actions_heatmap_scaled(actions_no_il)

alt.Chart(...)

In [34]:
actions_il = actions_df().filter(pl.col("state") == "IL")
actions_heatmap_scaled(actions_il)

alt.Chart(...)

#### 4.3 - Cumulative Line Chart

Another way to view this data would be with a cumulative line chart.

Create a chart with:

- days on the X axis
- cumulative actions to date on the Y axis
- one line per state

Hint: To do this you will need to look at the `transform_window` function.

See `ex4.3.png` for an example.

In [35]:
def actions_cumulative(df):
    # Ref for example implementation: https://altair-viz.github.io/gallery/line_chart_with_cumsum.html
    # Ref for grouping in window: https://altair-viz.github.io/user_guide/transform/window.html#user-guide-window-transform
    chart = alt.Chart(df).mark_line().transform_window(
        sort =[{'field': 'date_ymd'}],
        run_sum_actions = 'sum(actions)',
        days = 'count(date_ymd)',
        groupby=['state']
    ).encode(
        x='days:Q',
        y='run_sum_actions:Q',
        color='state:N'
    )

    return chart

# Note: I've included this by the number of days in session due to what
# seems like differences in the number of days in session. Since literal time
# limits are relevant for the total number of action, it seems "fairer" to
# compare based on session days, instead of calendar days.
actions_cumulative(actions_df())

alt.Chart(...)